# Elanco Students website: data pipeline demo
**Daluut Warsame · Data Analyst Interview Challenge 2026**

This notebook runs my cleaning pipeline and analysis end to end. Elanco's data isn't stored on GitHub, so step 1 uploads my submission zip, which contains it.

## 1. Load the project

In [ ]:
from google.colab import files
files.upload()   # choose Daluut_Warsame_Data_Analyst_Interview_Challenge.zip

In [ ]:
!unzip -q -o Daluut_Warsame_Data_Analyst_Interview_Challenge.zip
%cd Daluut_Warsame_Data_Analyst_Interview_Challenge

## 2. Clean and check the data
Reads the raw files without changing them, logs every fix with a count, then runs the quality checks. It stops if a serious check fails.

In [ ]:
!python src/pipeline.py

## 3. Daily summary
One row per day. Every count is of **distinct visits**, so repeated clicks in one visit count once.

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
daily = pd.read_csv('output/metrics/daily_summary.csv')
daily.head(10)

## 4. Breakdown by channel and device

In [ ]:
display(pd.read_csv('output/metrics/breakdown_channel.csv'))
display(pd.read_csv('output/metrics/breakdown_device.csv'))

## 5. Findings

In [ ]:
!python src/analysis.py > /dev/null
from IPython.display import Image, display
for c in ['01_daily_sessions','02_monthly_channel_mix','03_mobile_email_gap','04_apply_clicks_by_page','05_role_interest_share']:
    display(Image(f'reports/charts/{c}.png', width=900))

## 6. Why I count distinct visits
Joining visits to events and counting rows would more than double the real number of visits.

In [ ]:
s = pd.read_csv('data/raw/sessions.csv'); e = pd.read_csv('data/raw/events.csv')
print(f'{len(s):,} visits')
print(f'{len(e.merge(s, on="session_id")):,} rows after joining visits to events')

## 7. Why the bot traffic matters
The same days with the suspected bot sessions included: the email click rate collapses.

In [ ]:
print('Bots excluded:')
display(daily.query("date >= '2026-01-14' and date <= '2026-01-16'")[['date','eligible_sessions','application_email_click_rate']])
!sed -i 's/SUSPECTED_BOT_SOURCES = {"traffic-check.example"}/SUSPECTED_BOT_SOURCES = set()/' src/pipeline.py
!python src/pipeline.py > /dev/null
print('Bots included:')
display(pd.read_csv('output/metrics/daily_summary.csv').query("date >= '2026-01-14' and date <= '2026-01-16'")[['date','eligible_sessions','application_email_click_rate']])
!sed -i 's/SUSPECTED_BOT_SOURCES = set()/SUSPECTED_BOT_SOURCES = {"traffic-check.example"}/' src/pipeline.py
!python src/pipeline.py > /dev/null